# 02 — Raw SEEG Signal Inspection

## Objective

Load and inspect a representative intracranial EEG (SEEG) recording from the ds006519 dataset using MNE-Python.

This notebook focuses on understanding the raw time-series data before applying any preprocessing or signal transformations.

## Main steps

1. Load a BrainVision recording using MNE-Python.
2. Inspect the recording's properties and metadata.
3. Examine channel names and sampling frequency.
4. Extract and understand the numerical signal data.
5. Visualise selected SEEG channels in the time domain.

**Important:** The original recording will remain unchanged throughout this initial inspection.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
RAW_DATA_DIR = Path("../data/raw")

recording_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_ieeg.vhdr"
)

print(recording_path)
print(recording_path.exists())

In [ ]:
raw = mne.io.read_raw_brainvision(
    recording_path,
    preload=False,
    verbose=False
)

print(raw)

Loading the recording with MNE
- mne.io.read_raw_brainvision() loads BrainVision recordings into an MNE Raw object.
- The object contains signal data and recording metadata.
- preload=False delays loading the actual signal samples into memory.
- Representative recording: 31 channels, 131072 samples per channel, 32 seconds.
- At 4096 Hz, the total number of samples is consistent with the 32-second duration.

In [ ]:
print("Sampling frequency:", raw.info["sfreq"])
print("Number of channels:", raw.info["nchan"])
print("Channel names:", raw.ch_names)
print("Recording duration:", raw.times[-1], "seconds")

The channel names show (as the other metadata) that the channels are bipolar, which means that the voltage shown is the difference between two contacts.
Recording properties and channel organisation
- Sampling frequency: 4096 Hz.
- Number of channels: 31.
- Recording duration: 32 seconds.
- Number of samples per channel: 131072.
- Channels are bipolar SEEG derivations.
- Each channel measures the voltage difference between two adjacent electrode contacts.
- Channel names identify the two contacts involved in each bipolar derivation.
- Bipolar referencing changes the spatial interpretation of the recorded signal.

In [ ]:
print("Channel types:")
print(raw.get_channel_types())

print("\nMeasurement units:")
print(raw._orig_units) 

##results as seen on notebook 01, here type eeg bc of the library but in metadata seeg

Channel types and measurement units
- MNE initially classifies all 31 channels as eeg, although the dataset identifies them as SEEG.
- This is a software channel classification, not evidence that the signals are scalp EEG.
- Original measurement units: microvolts (µV).
- MNE internally represents EEG/SEEG amplitudes in volts (V).
- Each bipolar channel measures a voltage difference between two electrode contacts

In [ ]:
data = raw.get_data()


print("Data type:", type(data))
print("Array shape:", data.shape) ##(channels,samples)
print("Data units in MNE:", "Volts (V)")

In [ ]:
channel_index = 0

channel_data = data[channel_index, :]

print("Channel:", raw.ch_names[channel_index])
print("Number of samples:", len(channel_data))
print("Minimum amplitude:", np.min(channel_data), "V")
print("Maximum amplitude:", np.max(channel_data), "V")
print("Mean amplitude:", np.mean(channel_data), "V")
print("Standard deviation:", np.std(channel_data), "V")

Extracting signal data
- raw.get_data() retrieves the numerical signal samples as a NumPy array.
- Array dimensions: (n_channels, n_samples).
- For this recording: (31, 131072).
- Each row represents a bipolar SEEG channel.
- Each column represents one time sample.
- MNE returns signal amplitudes in volts.
- Basic descriptive statistics help characterise signal amplitude and variability.

Amplitude and descriptive statistics
- SEEG records voltage differences between electrode contacts; therefore, signal amplitude is measured in volts (V).
- MNE internally represents signal amplitudes in V.
- Conversion: \(1\,V = 10^6\,\mu V\).
- Mean, minimum, maximum and standard deviation have the same units as the original signal (V).
- Variance has squared units (V²).
- Standard deviation quantifies the spread of signal amplitudes around their mean.

In [ ]:
sfreq = raw.info["sfreq"]

times = np.arange(data.shape[1]) / sfreq

print("Number of time points:", len(times))
print("First time point:", times[0])
print("Last time point:", times[-1])

In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(times, channel_data)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (V)")
plt.title(f"Raw SEEG signal — {raw.ch_names[channel_index]}")

plt.grid(True)
plt.tight_layout()
plt.show()

Initial visual inspection of raw SEEG
- The signal fluctuates around zero during the pre- and post-stimulation periods.
- A prominent, high-amplitude, regular oscillation appears during electrical stimulation.
- This pattern is consistent with stimulation artefact rather than necessarily reflecting increased neural activity.
- Stimulation artefacts can dominate signal amplitude and spectral power.
- The dataset identifies two five-second analysis epochs: pre-stimulation and post-stimulation.
- Visual inspection is an initial quality-control step, not sufficient to establish signal quality or physiological effects.

## Extracting the pre-stimulation and post-stimulation epochs

The dataset provides two annotated five-second analysis epochs:

- Pre-stimulation: onset at 6.99823 seconds.
- Post-stimulation: onset at 19.05 seconds.

Both epochs are extracted from the continuous recording using their onset times and durations.

The extracted segments will be used for separate visual inspection and comparison.

In [ ]:
# Epoch onset times (seconds)
pre_start = 6.99823
post_start = 19.05

# Epoch duration (seconds)
epoch_duration = 5

# Convert times into sample indices
pre_start_sample = round(pre_start * sfreq)
post_start_sample = round(post_start * sfreq)

epoch_samples = int(epoch_duration * sfreq)

# Extract the two epochs
pre_data = data[
    :,
    pre_start_sample:pre_start_sample + epoch_samples
]

post_data = data[
    :,
    post_start_sample:post_start_sample + epoch_samples
]

print("Pre-stimulation shape:", pre_data.shape)
print("Post-stimulation shape:", post_data.shape)

In [ ]:
# Create a time axis relative to the beginning of each epoch
epoch_times = np.arange(epoch_samples) / sfreq

# Extract the same channel from both epochs
pre_channel = pre_data[channel_index, :]
post_channel = post_data[channel_index, :]

# Create two plots with shared axes
fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 7),
    sharex=True,
    sharey=True
)

# Plot pre-stimulation epoch
axes[0].plot(epoch_times, pre_channel)
axes[0].set_title("Pre-stimulation epoch")
axes[0].set_ylabel("Amplitude (V)")
axes[0].grid(True)

# Plot post-stimulation epoch
axes[1].plot(epoch_times, post_channel)
axes[1].set_title("Post-stimulation epoch")
axes[1].set_xlabel("Time within epoch (s)")
axes[1].set_ylabel("Amplitude (V)")
axes[1].grid(True)

fig.suptitle(f"SEEG signal comparison — {raw.ch_names[channel_index]}")
plt.tight_layout()
plt.show()

In [ ]:
print("PRE-STIMULATION")
print("Mean:", np.mean(pre_channel), "V")
print("Standard deviation:", np.std(pre_channel), "V")
print("Minimum:", np.min(pre_channel), "V")
print("Maximum:", np.max(pre_channel), "V")

print("\nPOST-STIMULATION")
print("Mean:", np.mean(post_channel), "V")
print("Standard deviation:", np.std(post_channel), "V")
print("Minimum:", np.min(post_channel), "V")
print("Maximum:", np.max(post_channel), "V")

In [ ]:
pre_slope = np.polyfit(epoch_times, pre_channel, 1)[0]
post_slope = np.polyfit(epoch_times, post_channel, 1)[0]

print("Pre-stimulation slope:", pre_slope, "V/s")
print("Post-stimulation slope:", post_slope, "V/s")

In [ ]:
# Calculate fitted linear trends
pre_trend = np.polyval(
    np.polyfit(epoch_times, pre_channel, 1),
    epoch_times
)

post_trend = np.polyval(
    np.polyfit(epoch_times, post_channel, 1),
    epoch_times
)

# Plot both signals and their fitted trends
fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 7),
    sharex=True
)

axes[0].plot(epoch_times, pre_channel, label="Raw signal")
axes[0].plot(epoch_times, pre_trend, label="Linear trend")
axes[0].set_title("Pre-stimulation")
axes[0].set_ylabel("Amplitude (V)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epoch_times, post_channel, label="Raw signal")
axes[1].plot(epoch_times, post_trend, label="Linear trend")
axes[1].set_title("Post-stimulation")
axes[1].set_xlabel("Time within epoch (s)")
axes[1].set_ylabel("Amplitude (V)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Preliminary observations

The raw SEEG signal from channel `B'02-B'03` was inspected over the complete recording and within the annotated pre- and post-stimulation epochs.

The two five-second epochs showed some differences:

- The post-stimulation epoch had greater signal variability, with a standard deviation of approximately 69.91 µV compared with 29.10 µV in the pre-stimulation epoch.
- Its peak-to-peak amplitude was approximately 327.00 µV, compared with 157.62 µV before stimulation.
- Linear trend estimation showed a positive slope in the post-stimulation epoch (+44.95 µV/s) and a slightly negative slope in the pre-stimulation epoch (-8.52 µV/s).

Both epoch means were relatively close to zero.

These observations suggest differences in signal amplitude variability and slow trends between the two epochs. However, they are preliminary descriptive findings from a single channel and recording. They do not establish a causal stimulation effect or a statistically significant difference.

## Limitations

- The signal has not yet undergone filtering or artefact removal.
- The analysis is limited to one channel and one recording.
- No frequency-domain or time-frequency analysis has been performed.
- The observed differences may reflect neural activity, recording artefacts, or other signal characteristics.

Further preprocessing and analysis are required before drawing physiological conclusions.